In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

In [ ]:
user_df = pd.read_csv("C:/Users/AANNYA/Desktop/DTI datasets/final_fintech_dataset_no_scores_no_recommendations (1).csv")

fd_df = pd.read_csv("C:/Users/AANNYA/Desktop/DTI datasets/india_property_fd_banks_2020_2026_dataset.csv")

property_df = pd.read_csv("C:/Users/AANNYA/Desktop/DTI datasets/india_required_states_property_dataset_2020_2026 (1).csv")

mf_df = pd.read_csv("C:/Users/AANNYA/Desktop/DTI datasets/mutual_funds_investment_dataset.csv")

print("All datasets loaded successfully")

In [ ]:
print(user_df.shape)
print(fd_df.shape)
print(property_df.shape)
print(mf_df.shape)

user_df.head()

In [ ]:
user_df["total_expenses"] = (
    user_df["rent"] +
    user_df["maintenance"] +
    user_df["emi"] +
    user_df["credit_card_payment"] +
    user_df["electricity"] +
    user_df["gas"] +
    user_df["internet"] +
    user_df["mobile"] +
    user_df["groceries"] +
    user_df["transportation"] +
    user_df["medical"] +
    user_df["household_items"]
)

In [ ]:
user_df["remaining_income"] = user_df["total_income"] - user_df["total_expenses"]

In [ ]:
def risk_profile(age):

    if age <= 25:
        return "High"

    elif age <= 40:
        return "Medium"

    else:
        return "Low"

In [ ]:
def recommend_sip(risk):

    funds = mf_df[mf_df["Risk_Level"] == risk]

    funds = funds.sort_values(
        by="Expected_Return_Percent",
        ascending=False
    )

    return list(funds["Fund_Type"].head(3).unique())

In [ ]:
def recommend_fd():

    latest = fd_df[fd_df["year"] == 2026]

    rates = {
        "SBI": latest["state_bank_of_india_fd_percent"].mean(),
        "HDFC": latest["hdfc_bank_fd_percent"].mean(),
        "ICICI": latest["icici_bank_fd_percent"].mean(),
        "Axis": latest["axis_bank_fd_percent"].mean(),
        "PNB": latest["punjab_national_bank_fd_percent"].mean(),
        "BOB": latest["bank_of_baroda_fd_percent"].mean(),
        "Canara": latest["canara_bank_fd_percent"].mean()
    }

    return max(rates, key=rates.get)

In [ ]:
def goal_investment_planner(income, expenses, age, goal_amount, months):

    remaining = income - expenses

    risk = risk_profile(age)

    monthly_goal = goal_amount / months

    print("\n------ Goal Based Investment AI ------")

    print("Goal Amount:", goal_amount)
    print("Time:", months, "months")

    print("\nMonthly Saving Required:", round(monthly_goal))

    if monthly_goal > remaining:

        deficit = monthly_goal - remaining

        print("\nGoal not possible with current spending")

        print("Reduce expenses by:", round(deficit))

        return

    # investment allocation

    if risk == "High":

        sip_ratio = 0.6
        fd_ratio = 0.25
        insurance_ratio = 0.15

    elif risk == "Medium":

        sip_ratio = 0.45
        fd_ratio = 0.35
        insurance_ratio = 0.20

    else:

        sip_ratio = 0.3
        fd_ratio = 0.45
        insurance_ratio = 0.25


    sip_amount = monthly_goal * sip_ratio
    fd_amount = monthly_goal * fd_ratio
    insurance_amount = monthly_goal * insurance_ratio


    sip_funds = recommend_sip(risk)
    fd_bank = recommend_fd()

    print("\nInvestment Strategy")

    print("Risk Profile:", risk)

    print("\nWhere To Invest")

    print("SIP Funds:", sip_funds)
    print("FD Bank:", fd_bank)

    print("\nMonthly Investment Plan")

    print("SIP:", round(sip_amount))
    print("FD:", round(fd_amount))
    print("Insurance:", round(insurance_amount))

In [ ]:
goal_investment_planner(
    income=70000,
    expenses=30000,
    age=25,
    goal_amount=20000,
    months=9
)